In [1]:
import joblib
import pandas as pd
import numpy as np

In [2]:
data = joblib.load("../data/processed/model_data.joblib")

In [3]:
X_train=data['X_train']
X_val = data['X_val']
X_test=data['X_test']

In [4]:
y_train=data['y_train']
y_val = data['y_val']
y_test = data['y_test']

In [5]:
print("Training size:",X_train.shape)
print("Validation shape: ", X_val.shape)
print("Testing shape: ", X_test.shape)

Training size: (71520, 2376)
Validation shape:  (15027, 2376)
Testing shape:  (15219, 2376)


In [6]:
#training decision tree

In [7]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score

In [8]:
dtree1 = DecisionTreeClassifier(random_state=42)
dtree1.fit(X_train,y_train)

y_pred = dtree1.predict(X_val) #predicting on validation data 

In [9]:
print("Accuracy: ",accuracy_score(y_val,y_pred))
print("Precision: ",precision_score(y_val,y_pred))
print("Recall: ",recall_score(y_val,y_pred))
print("F1 score: ",f1_score(y_val,y_pred))

Accuracy:  0.8240500432554735
Precision:  0.1697860962566845
Recall:  0.15338164251207728
F1 score:  0.1611675126903553


The basic decision tree achieved 82.41% accuracy, but its precision, recall, and F1-score were low. Since the dataset is imbalanced, accuracy alone is not enough to evaluate the model. The model missed many patients who were actually readmitted within 30 days.


In [10]:
from sklearn.metrics import confusion_matrix
coma = confusion_matrix(y_val,y_pred)
print(coma)

[[12129  1242]
 [ 1402   254]]


The model correctly identified 254 readmissions but missed 1,402 actual cases. It performed better at predicting patients who were not readmitted. This shows that the basic decision tree is not good at identifying the minority class.


training with class weights.
Since the dataset has fewer readmitted patients, I tried
class_weight='balanced' to give more importance to that class.
I will compare this model with the basic Decision Tree.


In [11]:
dtree2 = DecisionTreeClassifier(class_weight='balanced', random_state=42)
dtree2.fit(X_train,y_train)
y_pred2=dtree2.predict(X_val)

In [12]:
print("Accuracy: ",accuracy_score(y_val,y_pred2))
print("Precision: ",precision_score(y_val,y_pred2))
print("Recall: ",recall_score(y_val,y_pred2))
print("F1 score: ",f1_score(y_val,y_pred2))

Accuracy:  0.7941039462301192
Precision:  0.16433239962651727
Recall:  0.21256038647342995
F1 score:  0.18536071616640334


After using balanced class weights, recall increased from 15.34% to 21.26% and F1-score also improved. However, accuracy and precision decreased slightly. This shows that class weighting helped the model identify more readmissions, but the performance is still low.


##### we gonna train decision tree with limited depth

In [13]:
dtree3 = DecisionTreeClassifier(max_depth=5, min_samples_leaf=50,class_weight='balanced',random_state=42)
dtree3.fit(X_train,y_train)
y_pred3 = dtree3.predict(X_val)

In [14]:
print("Accuracy: ",accuracy_score(y_val,y_pred3))
print("Precision: ",precision_score(y_val,y_pred3))
print("Recall: ",recall_score(y_val,y_pred3))
print("F1 score: ",f1_score(y_val,y_pred3))

Accuracy:  0.7354761429427031
Precision:  0.1855166802278275
Recall:  0.41304347826086957
F1 score:  0.25603593486805165


I trained another Decision Tree using max_depth=5 and min_samples_leaf=50. Recall improved to 41.30% and F1-score increased to 25.60%. Accuracy decreased, but the model was able to identify more patients who were readmitted within 30 days.

##### Trying different tree depths

In [15]:
depths = [3,5,7,10,15]

for d in depths:
    dt = DecisionTreeClassifier(max_depth=d, min_samples_leaf=50, class_weight='balanced',random_state=42)
    dt.fit(X_train,y_train)
    prediction = dt.predict(X_val)
    
    recall = recall_score(y_val,prediction)
    f1 = f1_score(y_val,prediction)
    print("Depth:",d)
    print("Recall:",recall)
    print("F1:",f1)
    print("\n")
    
    

Depth: 3
Recall: 0.7397342995169082
F1: 0.24111800019683105


Depth: 5
Recall: 0.41304347826086957
F1: 0.25603593486805165


Depth: 7
Recall: 0.5380434782608695
F1: 0.2564766839378238


Depth: 10
Recall: 0.5295893719806763
F1: 0.2541292379020574


Depth: 15
Recall: 0.5543478260869565
F1: 0.23472257734594731




I tested tree depths of 3, 5, 7, 10, and 15. Depth 3 had the highest recall at 73.97%, while depth 7 had the highest F1-score at 25.65%. I will try different minimum leaf sizes before choosing the final model.


##### checking with different leaf sizes

In [16]:
leaf_sizes = [10,25,50,100,200]

for leaf in leaf_sizes:
    dt = DecisionTreeClassifier(max_depth=7, min_samples_leaf=leaf, class_weight='balanced',random_state=42)
    dt.fit(X_train,y_train)
    prediction = dt.predict(X_val)
    
    recall = recall_score(y_val,prediction)
    f1 = f1_score(y_val,prediction)
    print("Leaf:", leaf, "Recall:", recall, "F1:", f1)

Leaf: 10 Recall: 0.5428743961352657 F1: 0.2593768032313906
Leaf: 25 Recall: 0.538647342995169 F1: 0.25896356510378865
Leaf: 50 Recall: 0.5380434782608695 F1: 0.2564766839378238
Leaf: 100 Recall: 0.5301932367149759 F1: 0.2550472040668119
Leaf: 200 Recall: 0.5428743961352657 F1: 0.2593393913168902


I tested different minimum leaf sizes while keeping the tree depth at 7. Leaf size 10 gave the highest F1-score of 25.94% and recall of 54.29%. The results were quite similar, so I selected leaf size 10 for further evaluation.


#### evaluating our selected decision tree

In [17]:
best_dtree = DecisionTreeClassifier(max_depth=7, min_samples_leaf=10, class_weight='balanced', random_state=42)
best_dtree.fit(X_train,y_train)

best_pred = best_dtree.predict(X_val)

In [18]:
print("Accuracy: ",accuracy_score(y_val,best_pred))
print("Precision: ",precision_score(y_val,best_pred))
print("Recall: ",recall_score(y_val,best_pred))
print("F1 score: ",f1_score(y_val,best_pred))

Accuracy:  0.658348306381846
Precision:  0.1703942380591357
Recall:  0.5428743961352657
F1 score:  0.2593768032313906


In [19]:
print("confusion matrix is:")
print(confusion_matrix(y_val,best_pred))

confusion matrix is:
[[8994 4377]
 [ 757  899]]


After tuning the Decision Tree, recall increased to 54.29% and F1-score improved to 25.94%. The model correctly identified 899 readmissions but missed 757 cases. It also produced many false positives, so precision is still low.

In [20]:
## checking ROC-AUC curve
from sklearn.metrics import roc_auc_score, average_precision_score

probability = best_dtree.predict_proba(X_val)[:,1]
print("ROC-AUC:", roc_auc_score(y_val,probability))
print("average precision:", average_precision_score(y_val,probability))


ROC-AUC: 0.6476000362382068
average precision: 0.19014779071180515


The Decision Tree achieved a ROC-AUC of 0.6476 and an average precision of 0.1901. Both results are better than random prediction, but the model still has limited ability to identify readmitted patients.

In [22]:
thresholds=[0.3,0.4,0.5,0.6,0.7]

for t in thresholds:
    pred=(probability>=t).astype(int)
    p=precision_score(y_val,pred)
    r=recall_score(y_val,pred)
    f1=f1_score(y_val,pred)
    
    print("Threshold:",t)
    print("Precision:",p)
    print("Recall:",r)
    print("F1 score:",f1)
    print()
    

Threshold: 0.3
Precision: 0.11364906398435318
Recall: 0.982487922705314
F1 score: 0.20373153017781118

Threshold: 0.4
Precision: 0.1398670009811403
Recall: 0.7747584541062802
F1 score: 0.23695632098993444

Threshold: 0.5
Precision: 0.1703942380591357
Recall: 0.5428743961352657
F1 score: 0.2593768032313906

Threshold: 0.6
Precision: 0.2153972153972154
Recall: 0.3176328502415459
F1 score: 0.2567105905319668

Threshold: 0.7
Precision: 0.3149171270718232
Recall: 0.10326086956521739
F1 score: 0.15552523874488403



I tested thresholds from 0.3 to 0.7. Lower thresholds gave higher recall but lower precision. Threshold 0.5 had the highest F1-score of 25.94%, so I decided to keep it for the final model.

In [23]:
##testing decision tree

test_pred=best_dtree.predict(X_test)

In [24]:
print("Accuracy: ",accuracy_score(y_test,test_pred))
print("Precision: ",precision_score(y_test,test_pred))
print("Recall: ",recall_score(y_test,test_pred))
print("F1 score: ",f1_score(y_test,test_pred))

Accuracy:  0.6608844207898023
Precision:  0.17521925732412763
Recall:  0.5589285714285714
F1 score:  0.26679926125870157


In [25]:
print("confusion matrix:")
print(confusion_matrix(y_test,test_pred))

confusion matrix:
[[9119 4420]
 [ 741  939]]


The Decision Tree achieved 66.09% accuracy, 17.52% precision, 55.89% recall, and 26.68% F1-score on the test data. It correctly identified 939 readmissions but missed 741 cases. The results were close to the validation results, although the model still produced many false positives.

In [26]:
test_probability = best_dtree.predict_proba(X_test)[:,1]
print("ROC-AUC:",roc_auc_score(y_test,test_probability))
print("Average Precision:",average_precision_score(y_test,test_probability))

ROC-AUC: 0.6528723238686123
Average Precision: 0.20087937952720747


The Decision Tree achieved a ROC-AUC of 0.6529 and an average precision of 0.2009 on the test data. Both values were slightly higher than the validation results, but the model still has room for improvement.

In [27]:
##saving the decision tree model
import os

os.makedirs("../models", exist_ok=True)
joblib.dump(best_dtree,"../models/decision_tree.joblib")
print("Decision tree model is saved")

Decision tree model is saved
